# 🔥 Micrograd: See Forward Pass & Backprop, Don't Just Read About Them

Inspired directly by Andrej Karpathy's
[micrograd](https://github.com/karpathy/micrograd) and his video
["The spelled-out intro to neural networks and backpropagation"](https://www.classcentral.com/classroom/youtube-the-spelled-out-intro-to-neural-networks-and-backpropagation-building-micrograd-127040/695a09ede41b0).
Same idea, same teaching order, expanded with the theory/exercise format of this course.

**Why this actually works where formulas don't.** Forward pass and backprop are *hard to
digest* as equations because you can't see them happening. This notebook builds a tiny
autograd engine — about 40 lines of real code — from complete scratch, and **draws the
computation graph after every step** so you watch numbers flow forward and gradients flow
backward, on-screen, in real time. By the end you'll have built (by hand) the exact mechanism
that PyTorch's `autograd` and every deep learning framework runs under the hood.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ Rendered graph (not ASCII this time —
real computation-graph diagrams) → 🔬 Worked example → ⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn →
✅ Solution.

**Roadmap**
1. Forward pass, by hand, on paper-simple numbers
2. Wrapping numbers in a `Value` — and drawing the graph
3. Manual backprop on one multiplication (before any automation)
4. Manual backprop on a two-step expression (the chain rule, made visible)
5. Automating it: local gradient rules for `+`, `*`, `**`, `tanh`
6. The `backward()` method — topological sort, then one pass
7. Watching gradients flow: the full graph with data AND grad on every node
8. Building a Neuron, a Layer, and a tiny MLP out of `Value`
9. Training the MLP on a toy dataset — watch the loss curve fall
10. 🏆 Capstone: verify your engine against real PyTorch, prove it's correct


In [ ]:
# We'll use graphviz to actually DRAW the computation graph -- this is the
# single biggest difference from a textbook explanation: you SEE it.
import math
import random
from graphviz import Digraph
print("Ready.")

---
## Chapter 1 — Forward Pass, By Hand, On Paper-Simple Numbers

📖 **Theory.** The **forward pass** is just: plug numbers into an expression, left to right,
and write down every intermediate result. That's it. Nothing about neural networks is
conceptually different from this — a whole network is just a much bigger version of the tiny
expression below.

🧠 **Mental model.** Think of forward pass as a row of dominoes: each one's value depends only
on the ones before it. Compute them in order, left to right, and you're done.


In [ ]:
# The expression we'll use for the next few chapters: f = a*b + a
# Forward pass BY HAND, one line at a time, writing down every intermediate value
a = 2.0
b = -3.0

c = a * b       # step 1: c = 2 * -3 = -6
d = c + a       # step 2: d = -6 + 2 = -4

print(f"a = {a}")
print(f"b = {b}")
print(f"c = a*b = {c}")
print(f"d = c+a = {d}")
print(f"\nThat's the ENTIRE forward pass: {a} -> (times {b}) -> {c} -> (plus {a}) -> {d}")

### ✏️ Your Turn 1.1
By hand (then check with code), compute the forward pass of `f = (a + b) * b` for `a=3, b=4`.
Write down the intermediate value too, not just the final answer.

In [ ]:
a, b = 3.0, 4.0
intermediate = None   # a + b
final = None           # intermediate * b
print(intermediate, final)

✅ **Solution**
```python
intermediate = a + b   # 7.0
final = intermediate * b   # 28.0
```

---
## Chapter 2 — Wrapping Numbers in a `Value` — and Drawing the Graph

📖 **Theory.** Plain Python numbers forget where they came from — once you compute `c = a*b`,
`c` is just `-6.0`; there's no record that it came from multiplying `a` and `b`. To do
backprop, we need to **remember the whole history**: every number needs to know which
operation produced it and from which inputs. That's what the `Value` class does — it's a
number that **remembers its own computation graph**.

🧠 **Mental model.** A `Value` is a sticky note with a number on it, plus a little arrow
pointing back to whatever sticky notes and operation made it. Chain enough sticky notes
together and you have the whole graph.


In [ ]:
class Value:
    """A scalar that remembers HOW it was computed -- the entire trick."""
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)   # which Values made this one
        self._op = _op                 # which operation made this one ('+', '*', ...)

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

    def __repr__(self):
        return f"Value(data={self.data})"

a = Value(2.0)
b = Value(-3.0)
c = a * b
d = c + a
print("d =", d)
print("d remembers it came from:", d._op, "of", [v.data for v in d._prev])

⚡ **Pro tip.** This is *exactly* how PyTorch tensors work under the hood — every tensor
with `requires_grad=True` is secretly a `Value`-like object with a `.grad_fn` pointing back
to whatever created it. You've now seen the actual mechanism, not just the API.

### ✏️ Your Turn 2.1
Create `x = Value(5.0)` and `y = Value(2.0)`, compute `z = x * y`, and print what operation
and which values `z` remembers being built from.

In [ ]:
x = Value(5.0)
y = Value(2.0)
z = None
print(z._op, [v.data for v in z._prev])

✅ **Solution**
```python
z = x * y
print(z._op, [v.data for v in z._prev])   # '*' [5.0, 2.0]
```

### Let's actually SEE it

This is the payoff for wrapping numbers in `Value`: now we can draw the graph. Every box is a
`Value` (its data), every oval is an operation. Run this and look at the picture, not just
the numbers.


In [ ]:
def trace(root):
    """Walk the graph backward from root, collecting every node and edge."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_graph(root, show_grad=False):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        label = f"data {n.data:.4f}"
        if show_grad:
            label += f"\ngrad {n.grad:.4f}"
        dot.node(name=uid, label=label, shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op, shape='circle')
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

a = Value(2.0); b = Value(-3.0)
c = a * b
d = c + a
draw_graph(d)

### ✏️ Your Turn 2.2
Build the graph for `f = (a + b) * a` with `a=3, b=-1` and render it with `draw_graph`. Before
running it, guess: how many operation-ovals will the picture have?

In [ ]:
a = Value(3.0); b = Value(-1.0)
f = None
draw_graph(f)

✅ **Solution**
```python
f = (a + b) * a
draw_graph(f)   # 2 operation ovals: '+' then '*'
```

---
## Chapter 3 — Manual Backprop on ONE Multiplication (Before Any Automation)

📖 **Theory.** Before we automate anything, let's do the *simplest possible* backward pass
completely by hand, so the automation in Chapter 5 isn't magic. For `c = a * b`, we want to
know: **if `c` changes by a tiny amount, how much does that "blame" `a` and `b` for it?**

The answer for multiplication: `dc/da = b` and `dc/db = a`. (This is just the power/product
rule from calculus — see the Math & Statistics Foundations guide if you want the formal
derivation.)

🧠 **Mental model.** Each input's gradient answers: **"if I nudge just this input a tiny bit,
how much does the output move?"** For `c = a*b`, nudging `a` by a tiny epsilon moves `c` by
`b * epsilon` — so `a`'s "sensitivity" is exactly `b`.


In [ ]:
a_val, b_val = 2.0, -3.0
c_val = a_val * b_val
print(f"c = a*b = {c_val}")

# by hand: dc/da = b, dc/db = a
dc_da = b_val
dc_db = a_val
print(f"dc/da = {dc_da}  (== b)")
print(f"dc/db = {dc_db}  (== a)")

# verify numerically with a tiny nudge (this is the DEFINITION of a derivative)
eps = 0.0001
c_nudged = (a_val + eps) * b_val
numerical_slope = (c_nudged - c_val) / eps
print(f"\nnumerical check: (c(a+eps) - c(a)) / eps = {numerical_slope:.4f}  (should match dc/da)")

⚡ **Pro tip.** That "nudge by epsilon and measure the change" trick is called **numerical
gradient checking** — it's how you'd verify ANY gradient implementation is correct, including
in real production code. We'll use it again in the capstone.

### ✏️ Your Turn 3.1
For `c = a + b` (addition, not multiplication), what are `dc/da` and `dc/db`? Verify with the
same epsilon-nudge trick.

In [ ]:
a_val, b_val = 4.0, 7.0
c_val = a_val + b_val
dc_da = None   # your guess
dc_db = None   # your guess
eps = 0.0001
numerical_slope = ((a_val+eps) + b_val - c_val) / eps
print(dc_da, dc_db, numerical_slope)

✅ **Solution**
```python
dc_da = 1.0   # nudging a by eps moves c by exactly eps -> slope 1
dc_db = 1.0   # same for b, by symmetry
# numerical_slope should print ~1.0, confirming it
```

---
## Chapter 4 — Manual Backprop on TWO Steps (the Chain Rule, Made Visible)

📖 **Theory.** Now the real question: for `d = c + a` where `c = a * b`, what is `dd/da`?
Notice `a` affects `d` **two ways** — directly (through the `+ a`) and indirectly (through
`c = a*b`). The chain rule says: **add up every path's contribution.**

```
dd/da = (dd/dc * dc/da)   +   (dd/da direct)
       = (1 * b)           +   1
```

🖼️ Picture this on the graph you drew in Chapter 2: `a` has TWO arrows leaving it (one into
the `*`, one into the `+`). Backprop must walk BOTH arrows and add up what comes back along
each one — that's the whole chain rule, made visual.


In [ ]:
a_val, b_val = 2.0, -3.0
c_val = a_val * b_val      # -6
d_val = c_val + a_val      # -4

# manual chain rule: dd/da has TWO contributions
dd_dc = 1.0          # local: d = c + a, so dd/dc = 1
dc_da = b_val         # local: c = a*b, so dc/da = b

path_through_c = dd_dc * dc_da   # contribution via c
path_direct = 1.0                 # contribution via the direct '+a' term

total_dd_da = path_through_c + path_direct
print(f"contribution via c: {path_through_c}")
print(f"contribution direct: {path_direct}")
print(f"TOTAL dd/da = {total_dd_da}")

# numerical check
eps = 0.0001
d_nudged = (a_val+eps)*b_val + (a_val+eps)
numerical = (d_nudged - d_val) / eps
print(f"numerical check: {numerical:.4f}")

⚠️ **Common trap.** This is THE most common backprop bug, in real code and by hand:
forgetting that a value used in **multiple places** must have its gradient contributions
**added together**, not overwritten. Miss this and gradients are silently wrong. This is
exactly why the automated version (next chapter) uses `+=` for every gradient update, never `=`.

### ✏️ Your Turn 4.1
For `d = c + a` (same as above), what's `dd/db`? (Hint: does `b` have one path to `d` or two?)

In [ ]:
dd_db = None
print(dd_db)

✅ **Solution**
```python
# b only affects d through c = a*b (ONE path), so:
dd_db = dd_dc * a_val   # = 1 * 2 = 2.0
```

---
## Chapter 5 — Automating It: Local Gradient Rules Per Operation

📖 **Theory.** Every operation needs to know only ONE thing to participate in backprop: **its
own local derivative rule.** `+` needs to know `d(a+b)/da = 1`. `*` needs to know `d(a*b)/da =
b`. Each `Value` will store a tiny `_backward` function that knows how to push gradient
backward through JUST that one operation — and, critically, **adds** (`+=`) to each input's
grad, never overwrites, per the trap in Chapter 4.

🧠 **Mental model.** Each operation is a tiny, self-contained "gradient router" — it only
needs to know how to route gradient to ITS OWN inputs. Chain enough of these together and the
full chain rule happens automatically, with no operation needing to know about the others.


In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0                 # NEW: accumulates gradient here
        self._backward = lambda: None    # NEW: how to push gradient to children
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad     # local derivative of + is 1
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad   # local derivative: d(a*b)/da = b
            other.grad += self.data * out.grad    # d(a*b)/db = a
        out._backward = _backward
        return out

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

# Try it: set the OUTPUT's grad to 1 (a value is always "fully responsible for itself"),
# then manually fire the _backward functions in the right order (we'll automate the
# ORDER in the next chapter -- for now, do it by hand to see each piece work).
a = Value(2.0); b = Value(-3.0)
c = a * b
d = c + a

d.grad = 1.0        # d/dd(d) = 1, always, for whatever you call .backward() on
d._backward()        # pushes gradient from d into c and a (the '+' operation's local rule)
c._backward()        # pushes gradient from c into a and b (the '*' operation's local rule)

print("a.grad =", a.grad, " (should be -2.0, matching Chapter 4)")
print("b.grad =", b.grad, " (should be 2.0, matching Chapter 4's Your Turn)")

⚠️ **Common trap.** Notice we called `d._backward()` **before** `c._backward()` — that
order matters! `c`'s backward needs `c.grad` to already be fully accumulated (from `d`'s
backward), or it'll push forward an incomplete/wrong gradient. Getting this order right,
automatically, for any graph shape, is exactly what Chapter 6 solves.

### ✏️ Your Turn 5.1
Add a `__pow__` method to a NEW class (or extend this one) implementing `x ** n` for a
constant integer `n`, with the correct local derivative `d(x^n)/dx = n * x^(n-1)`.

In [ ]:
def add_pow_method():
    pass  # describe/write the __pow__ method here
# reference implementation to check yourself against:
'''
def __pow__(self, other):
    assert isinstance(other, (int, float))
    out = Value(self.data ** other, (self,), f'**{other}')
    def _backward():
        self.grad += (other * self.data**(other-1)) * out.grad
    out._backward = _backward
    return out
'''

✅ **Solution** (shown above as reference) — the pattern is always the same: compute the
forward value, remember the local derivative rule, wire it into `_backward`.

---
## Chapter 6 — The `backward()` Method: Topological Sort, Then One Pass

📖 **Theory.** Chapter 5 called `_backward()` by hand, in the right order, because we could
see the tiny graph. For an arbitrary graph (like a whole neural network), we need to
**compute** the right order automatically: every node's `_backward()` must run only AFTER
every node that USES it has already run theirs. This ordering is called a **topological
sort**, and it's a real, well-known graph algorithm (also covered in the Algorithms &
Data Structures section of this course) — not something special to neural networks.

🖼️ **The algorithm in one sentence:** do a depth-first walk from the output, and add each
node to a list only AFTER all its children are already in the list — then process that list
**in reverse**.


In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # 1. topological sort: build an order where every node comes AFTER its children
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        # 2. seed: the output is "fully responsible for itself"
        self.grad = 1.0

        # 3. walk the list BACKWARD, firing each node's local backward rule
        for v in reversed(topo):
            v._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

# The full pipeline, now ONE method call instead of manual ordering:
a = Value(2.0); b = Value(-3.0)
c = a * b
d = c + a
d.backward()
print("a.grad =", a.grad, " b.grad =", b.grad, "  (matches Chapters 4-5, now automatic)")

### ✏️ Your Turn 6.1
Build `f = a**2 + b**2` for `a=3, b=4` (this is literally the Euclidean distance formula
squared, from the NumPy & Linear Algebra guide!), call `f.backward()`, and check `a.grad`
and `b.grad` against the known derivative `df/da = 2a`.

In [ ]:
a = Value(3.0); b = Value(4.0)
f = None
f.backward()
print(a.grad, b.grad)   # expect 6.0 and 8.0

✅ **Solution**
```python
f = a**2 + b**2
f.backward()
print(a.grad, b.grad)   # 6.0, 8.0 -- matches 2*3 and 2*4
```

---
## Chapter 7 — Watching Gradients Flow: Data AND Grad, On the Same Graph

📖 **Theory.** Now for the payoff. We already built `draw_graph()` in Chapter 2. Call it again
AFTER `.backward()`, with `show_grad=True` — and you'll see the **exact same picture**, now
annotated with every gradient, all at once. This is precisely the diagram Karpathy draws by
hand on a whiteboard in the video — except yours is generated from a REAL, RUNNING engine you
built yourself.

⚡ **Pro tip.** Read the graph right-to-left for forward values (data), then left-to-right for
gradients (grad) — that's forward pass and backward pass, literally visible as two directions
through the same picture.


In [ ]:
a = Value(2.0); b = Value(-3.0); c_input = Value(10.0)
e = a * b
d = e + c_input
f_val = Value(-2.0)
L = d * f_val

L.backward()
draw_graph(L, show_grad=True)

### ✏️ Your Turn 7.1
Build a slightly bigger graph: `L = (a*b + c) * (a - b)` for `a=2, b=-3, c=10`. Call
`.backward()` and render it with grads shown. Before looking at the picture, predict: does `a`
have more than one arrow leaving it? (Yes — it's used twice, just like in Chapter 4.)

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
L = None
L.backward()
draw_graph(L, show_grad=True)

✅ **Solution**
```python
L = (a*b + c) * (a - b)
L.backward()
draw_graph(L, show_grad=True)
# a's picture shows two outgoing arrows -- into the '*' AND into the '-' --
# exactly the "used in two places -> gradients add up" trap from Chapter 4.
```

---
## Chapter 8 — Building a Neuron, a Layer, and a Tiny MLP

📖 **Theory.** A single artificial neuron (from the Neural Networks concept guide) is:
`output = activation(w1*x1 + w2*x2 + ... + b)`. We now have every piece needed to build this
with `Value`s: multiplication, addition, and `tanh` as the activation — and because every
piece is a `Value`, **the ENTIRE network's forward pass automatically builds one giant
computation graph**, and `.backward()` computes every weight's gradient with no extra code.

🧠 **Mental model.** A neuron is just the two-step expression from Chapter 4, repeated once
per input and summed, then squashed through `tanh`. A layer is several neurons run on the
same inputs. An MLP is layers chained together — each one's output feeding the next one's
input, exactly like the domino chain from Chapter 1, just wider and deeper.


In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w1*x1 + w2*x2 + ... + b, then squash through tanh -- exactly the neuron formula
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(42)
mlp = MLP(3, [4, 4, 1])   # 3 inputs -> hidden(4) -> hidden(4) -> 1 output
x = [Value(2.0), Value(3.0), Value(-1.0)]
out = mlp(x)
print("MLP output:", out)
print("total learnable parameters:", len(mlp.parameters()))

⚡ **Pro tip.** Count the parameters: `3->4->4->1` should give `(3*4+4) + (4*4+4) + (4*1+1) =
16 + 20 + 5 = 41`. This is the exact same "weights and biases" counting you'd do reading a
`torchinfo.summary()` output in the PyTorch lab — same concept, now built by hand.

### ✏️ Your Turn 8.1
Build an `MLP(2, [4, 1])` (2 inputs, one hidden layer of 4, one output) and run it on
`x = [Value(1.0), Value(-1.0)]`. How many total parameters does it have?

In [ ]:
random.seed(0)
mlp2 = MLP(2, [4, 1])
x2 = [Value(1.0), Value(-1.0)]
out2 = mlp2(x2)
n_params = None
print(out2, n_params)

✅ **Solution**
```python
n_params = len(mlp2.parameters())   # (2*4+4) + (4*1+1) = 12 + 5 = 17
```

---
## Chapter 9 — Training the MLP: Watch the Loss Curve Fall

📖 **Theory.** Training is the exact 5-step rhythm from the PyTorch lab, just written by
hand: forward pass (get predictions) → compute loss → `loss.backward()` → nudge every
parameter opposite its gradient → repeat. The ONLY new code here is the training loop itself
— every piece it calls (`Value`, `.backward()`, `MLP`) you already built.


In [ ]:
# A tiny toy dataset: 4 points in 2D, binary-ish targets (-1 or 1)
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]   # desired outputs

random.seed(1)
mlp = MLP(3, [4, 4, 1])
losses = []

for step in range(50):
    # forward pass: run every example through the network
    y_pred = [mlp(x)[0] if isinstance(mlp(x), list) else mlp(x) for x in xs]
    y_pred = [mlp(x) for x in xs]

    # loss: mean squared error, built entirely out of Value operations --
    # so the loss itself is just one more node in the SAME computation graph
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, y_pred))

    # zero gradients (same reason as PyTorch: grads ACCUMULATE with +=, must reset)
    for p in mlp.parameters():
        p.grad = 0.0

    # backward pass -- ONE call computes every parameter's gradient
    loss.backward()

    # update: nudge every parameter opposite its gradient (gradient descent, by hand)
    learning_rate = 0.05
    for p in mlp.parameters():
        p.data -= learning_rate * p.grad

    losses.append(loss.data)
    if step % 10 == 0:
        print(f"step {step:2d}  loss {loss.data:.4f}")

print(f"\nfinal loss: {losses[-1]:.4f}  (started at {losses[0]:.4f})")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,3))
plt.plot(losses)
plt.xlabel("training step"); plt.ylabel("loss"); plt.title("Loss falling -- backprop actually learning")
plt.show()

### ✏️ Your Turn 9.1
Predict what each example's output SHOULD be close to (`ys` above), then print the trained
network's actual predictions for all 4 examples and compare.

In [ ]:
final_preds = [mlp(x) for x in xs]
for x, y_true, y_pred in zip(xs, ys, final_preds):
    print(f"target={y_true:+.1f}  predicted={y_pred.data:+.4f}")

✅ **Solution** — just run the cell above; after 50 steps, each prediction should be
noticeably closer to its target than a random guess would be (predictions near +1 for
targets of 1.0, near -1 for targets of -1.0).

---
## 🏆 Chapter 10 — Capstone: Verify Your Engine Against Real PyTorch

The ultimate proof that you've genuinely built a correct autograd engine: run the **exact
same expression** through your `Value` class and through real PyTorch, and confirm the
gradients match **exactly**. This is the same numerical-gradient-checking spirit from
Chapter 3, now checking your whole engine against the real thing.

In [ ]:
import torch

# The same expression in BOTH engines
def run_micrograd():
    a = Value(-4.0)
    b = Value(2.0)
    c = a + b
    d = a * b + b**3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).tanh()
    d = d + 3 * d + (b - a).tanh()
    e = c - d
    f = e**2
    g = f / 2.0
    g = g + 10.0 / f
    g.backward()
    return g.data, a.grad, b.grad

def run_pytorch():
    a = torch.Tensor([-4.0]).double(); a.requires_grad = True
    b = torch.Tensor([2.0]).double();  b.requires_grad = True
    c = a + b
    d = a * b + b**3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).tanh()
    d = d + 3 * d + (b - a).tanh()
    e = c - d
    f = e**2
    g = f / 2.0
    g = g + 10.0 / f
    g.backward()
    return g.data.item(), a.grad.item(), b.grad.item()

### ✏️ Capstone Task
Run both functions and compare all three values (`g`, `a.grad`, `b.grad`) between engines.
They should match to many decimal places -- that's not a coincidence, it's proof your
40-line engine implements the exact same mathematics as a production framework.

In [ ]:
mg_g, mg_da, mg_db = None, None, None
pt_g, pt_da, pt_db = None, None, None
print("micrograd:", mg_g, mg_da, mg_db)
print("pytorch:  ", pt_g, pt_da, pt_db)

✅ **Capstone Solution**
```python
mg_g, mg_da, mg_db = run_micrograd()
pt_g, pt_da, pt_db = run_pytorch()

print("micrograd:", mg_g, mg_da, mg_db)
print("pytorch:  ", pt_g, pt_da, pt_db)

assert abs(mg_g - pt_g) < 1e-6
assert abs(mg_da - pt_da) < 1e-6
assert abs(mg_db - pt_db) < 1e-6
print("\\nMATCH -- your from-scratch engine computes IDENTICAL gradients to PyTorch.")
```

🎉 **You've built and verified a real autograd engine from scratch.** Forward pass is just
evaluating an expression and remembering the graph. Backward pass is just: seed the output's
grad as 1, then walk the graph in reverse topological order, letting each operation's local
derivative rule push gradient to its inputs, ADDING when a value is used more than once. That
is the entire idea — everything else (PyTorch, TensorFlow, the Transformer attention you built
in an earlier lab) is this same mechanism at a much larger scale, with more operations and
way more nodes, but not a different idea.

---
### 📌 Concept Quick-Reference
**Forward pass:** evaluate an expression left to right, remembering every intermediate value
**Value class:** a number + which operation + which inputs made it (the computation graph)
**Local derivative:** each op only needs to know ITS OWN rule (`+`->1, `*`->other input, etc.)
**Chain rule, visually:** a node used in multiple places has multiple outgoing graph edges;
  gradient contributions from each edge are ADDED (`+=`), never overwritten
**Topological sort:** the algorithm that finds a valid "process children before parents" order
**backward():** seed output.grad=1, walk topo order in reverse, fire each node's local rule
**Neuron/Layer/MLP:** built entirely from Value operations -> gradients come for free
**Training loop:** forward -> loss -> zero_grad -> backward -> nudge params -> repeat
**Sanity check:** compare against numerical gradients (tiny nudge) or a real framework (PyTorch)

### 📌 Where to go next
- Rewatch Karpathy's video now — every line of code in it will make sense, because you just
  wrote (a course-adapted version of) all of it yourself.
- `Concept_Guides/04_Neural_Networks_Backpropagation_Concept_Guide.pdf` — the same ideas,
  formula-first, for a quick re-read.
- `Zero_to_Hero_Series/04_PyTorch_Zero_to_Hero/` — see this exact mechanism, scaled up,
  under PyTorch's real `autograd`.
- `Zero_to_Hero_Series/11_Transformers_Attention_Zero_to_Hero/` — the same backward pass,
  now flowing through attention instead of a tiny MLP.
